## PROYECTO: PORTAL DE TICKETS PQRS
### ETAPA: TRANSFORMACIÓN (CAPA SILVER)
**OBJETIVO:** Limpiar, cruzar tablas (Tickets + Agentes) y aplicar reglas de negocio para enriquecer los datos.

In [0]:
from pyspark.sql.functions import col, current_timestamp, when
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
    
# Limpiamos widgets previos para evitar conflictos
dbutils.widgets.removeAll()

In [0]:
# 1. PARÁMETROS DINÁMICOS (WIDGETS)

dbutils.widgets.text("catalogo", "catalogo_pqrs")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

print("✅ Parámetros cargados.")

In [0]:
# 2. LECTURA DE TABLAS DESDE BRONZE

df_tickets = spark.table(f"{catalogo}.{esquema_source}.tickets_raw")
df_agentes = spark.table(f"{catalogo}.{esquema_source}.agentes_soporte_raw")

In [0]:
#3. CREACIÓN DE FUNCIÓN PERSONALIZADA (UDF)
# Regla de negocio: Categorizar la criticidad basada en el estado del ticket

def evaluar_criticidad(estado):
    if estado in ["Abierto", "Escalado"]:
        return "Alta"
    elif estado == "En Proceso":
        return "Media"
    else:
        return "Baja"

criticidad_udf = F.udf(evaluar_criticidad, StringType())

In [0]:
# 4. LIMPIEZA Y TRANSFORMACIÓN
# Eliminamos nulos en llaves críticas y aplicamos la UDF de criticidad

df_tickets_clean = df_tickets.dropna(how="any", subset=["ticket_id", "agente_id"]) \
                             .withColumn("criticidad", criticidad_udf(col("estado")))

In [0]:
# 5. CRUCE DE TABLAS (JOIN)
# Cruzamos los tickets con la información del agente asignado

df_joined = df_tickets_clean.alias("t").join(
    df_agentes.alias("a"),
    col("t.agente_id") == col("a.id_agente"),
    "left"
)

In [0]:
# 6. SELECCIÓN FINAL Y GUARDADO EN SILVER

tabla_destino = f"{catalogo}.{esquema_sink}.tickets_enriquecidos"

df_silver = df_joined.select(
    col("t.id_ticket"),
    col("t.fecha_creacion"),
    col("t.estado"),
    col("criticidad"),
    col("a.nombre_completo").alias("agente_asignado"),
    col("a.region").alias("region_agente"),
    col("a.nivel_soporte"),
    current_timestamp().alias("fecha_transformacion")
)

df_silver.write \
    .mode("overwrite") \
    .saveAsTable(tabla_destino)

print(f"✅ Transformación completada. Tabla guardada en {tabla_destino}")